# 46 — Tree of Thoughts

## What you'll learn

- **Send API for parallel branch fan-out** — how `list[Send]` from a conditional entry point spawns N independent node invocations simultaneously
- **`Annotated[list, operator.add]` reducer pattern** — accumulating results from parallel branches back into the root state without race conditions
- **Separate generator vs judge LLMs** — using high temperature for creative thought generation and low temperature for consistent 0-10 scoring
- **`set_conditional_entry_point` with `list[Send]` return** — the LangGraph idiom for fan-out at the graph entry point

## Context

Tree of Thoughts (Yao et al., 2023) improves LLM reasoning by exploring multiple thought paths instead of committing to a single chain of thought. Instead of greedy left-to-right generation, ToT fans out N candidate thoughts, evaluates each, and selects the most promising path — mimicking how humans explore and backtrack when solving hard problems.

LangGraph's Send API makes this a first-class pattern: `Send("node_name", state_dict)` schedules an independent invocation of any node with its own private state, and an `Annotated` reducer on the parent state merges the results as branches complete.

In [ ]:
# Install dependencies (run once)
import subprocess, sys
pkgs = ["langchain", "langchain-openai", "langgraph", "python-dotenv"]
missing = []
for pkg in pkgs:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        missing.append(pkg)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print(f"Installed: {missing}")
else:
    print("All dependencies already installed")

In [ ]:
# API key setup
import os
from dotenv import load_dotenv

load_dotenv()  # loads from .env if present locally

# In Colab: set via Secrets (key icon in sidebar) or directly:
# os.environ["OPENAI_API_KEY"] = "sk-..."

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY not set. Add it to .env or set os.environ['OPENAI_API_KEY']")
print("API key loaded")

In [ ]:
# Concept: Chain of Thought vs Tree of Thoughts

# (1) Chain of Thought: single linear path — greedy, committed
# START → thought_1 → thought_2 → thought_3 → answer
#   Problem: one bad step poisons the chain

# (2) Tree of Thoughts: fan-out → score → select → synthesize (best-first)
# START
#   ├─ Send → branch_0 (score=7) ─┐
#   ├─ Send → branch_1 (score=9) ─┤→ select best → synthesize → answer
#   └─ Send → branch_2 (score=5) ─┘

# The Annotated reducer pattern that makes fan-out work:
#
#   from typing import Annotated
#   import operator
#
#   class ToTState(TypedDict):
#       branches: Annotated[list[dict], operator.add]   # <-- reducer
#
# Each branch returns {"branches": [{"branch_id": i, "thought": ..., "score": ...}]}
# LangGraph applies operator.add (list concatenation) so all results accumulate safely.

print("ToT: fan out N branches, score each, select best")
print("Key: Annotated[list[dict], operator.add] accumulates parallel results")
print("Key: Send('node', state) spawns an independent node invocation")

In [ ]:
# Full self-contained Tree of Thoughts implementation
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from langgraph.types import Send

N_BRANCHES = 3

# --- State definitions ---

class BranchState(TypedDict):
    """Private state for each parallel thought branch."""
    problem: str
    branch_id: int
    thought: str
    score: int

class ToTState(TypedDict):
    """Root state shared across the graph."""
    problem: str
    branches: Annotated[list[dict], operator.add]  # accumulates all branch results
    best_thought: str
    final_answer: str

# --- LLMs ---
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.8)       # diverse thought generation
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1) # consistent scoring

# --- Prompts ---
BRANCH_PROMPT = """You are exploring solution path {branch_id} of {n_branches} for this problem.
Approach it from a DIFFERENT angle than typical solutions.

Problem: {problem}

Generate a creative, well-reasoned thought (3-5 sentences). Be specific and concrete."""

SCORE_PROMPT = """Rate this thought on a scale of 0-10 for solving the problem.
Criteria: relevance (0-4), creativity (0-3), feasibility (0-3).

Problem: {problem}
Thought: {thought}

Respond with ONLY a number 0-10."""

SYNTHESIS_PROMPT = """You selected the best thought from a tree search. Now expand it into a complete answer.

Problem: {problem}
Best Thought: {thought}
(This was scored highest of {n} candidate thoughts)

Write a thorough, actionable response."""

# --- Nodes ---

def generate_branches(state: ToTState) -> list[Send]:
    """Fan out N branches using the Send API."""
    return [
        Send("score_branch", {"problem": state["problem"], "branch_id": i, "thought": "", "score": 0})
        for i in range(N_BRANCHES)
    ]

def score_branch(state: BranchState) -> dict:
    """Generate thought for this branch, then score it."""
    branch_prompt = BRANCH_PROMPT.format(
        branch_id=state["branch_id"] + 1,
        n_branches=N_BRANCHES,
        problem=state["problem"],
    )
    thought_result = llm.invoke([HumanMessage(content=branch_prompt)])
    thought = thought_result.content.strip()

    score_prompt = SCORE_PROMPT.format(problem=state["problem"], thought=thought)
    score_result = judge_llm.invoke([HumanMessage(content=score_prompt)])
    try:
        score = int("".join(filter(str.isdigit, score_result.content.strip()))[:2])
        score = min(10, max(0, score))
    except (ValueError, IndexError):
        score = 5
    print(f"  Branch {state['branch_id'] + 1}: score={score}/10 | {thought[:70]}...")
    return {"branches": [{"branch_id": state["branch_id"], "thought": thought, "score": score}]}

def select_best(state: ToTState) -> dict:
    """Pick highest-scoring branch."""
    best = max(state["branches"], key=lambda b: b["score"])
    print(f"  Best branch: {best['branch_id'] + 1} (score={best['score']}/10)")
    return {"best_thought": best["thought"]}

def synthesize(state: ToTState) -> dict:
    """Expand best thought into final answer."""
    prompt = SYNTHESIS_PROMPT.format(
        problem=state["problem"],
        thought=state["best_thought"],
        n=N_BRANCHES,
    )
    result = llm.invoke([HumanMessage(content=prompt)])
    return {"final_answer": result.content}

# --- Graph ---

def create_workflow():
    graph = StateGraph(ToTState)
    graph.add_node("score_branch", score_branch)
    graph.add_node("select_best", select_best)
    graph.add_node("synthesize", synthesize)
    graph.set_conditional_entry_point(generate_branches, {"score_branch": "score_branch"})
    graph.add_edge("score_branch", "select_best")
    graph.add_edge("select_best", "synthesize")
    graph.add_edge("synthesize", END)
    return graph.compile()

app = create_workflow()
print("Workflow compiled successfully")

In [ ]:
# Run 2 problems and display results
SAMPLE_PROBLEMS = [
    "Design a system to recommend personalized learning paths for software engineers.",
    "How should a startup prioritize features when resources are limited?",
]

results = []

for problem in SAMPLE_PROBLEMS:
    print(f"\nPROBLEM: {problem}")
    print("-" * 65)
    result = app.invoke({"problem": problem, "branches": [], "best_thought": "", "final_answer": ""})
    results.append(result)

    # Branch score table
    print(f"\n{'Branch':<10} {'Score':>8} {'Thought preview':<50}")
    print("-" * 70)
    for b in sorted(result["branches"], key=lambda x: x["score"], reverse=True):
        winner = " <- best" if b["thought"] == result["best_thought"] else ""
        preview = b["thought"][:45] + "..."
        print(f"  Branch {b['branch_id']+1:<4} {b['score']:>6}/10   {preview}{winner}")

    print(f"\nFinal Answer preview:")
    print(result["final_answer"][:400])
    print()

## Exercises

1. **Change `N_BRANCHES` to 5** — does the quality of the final answer improve? By how much? At what cost in LLM calls?

2. **Add `beam_width=2`** — instead of selecting a single best branch, keep the top-2 branches and pass both to `synthesize`. Modify `select_best` to return a list and update `synthesize` to incorporate multiple thoughts.

3. **Add a second round of branching from the best thought (2-level tree)** — after `select_best`, fan out again with 3 sub-branches that extend the best thought, score those, and pick the best sub-thought before synthesizing.

4. **Replace the judge LLM with a Python heuristic** — score thoughts by word count, presence of domain keywords, or sentence count. When is a heuristic sufficient, and when does the judge LLM add real value?

---

## What's next

- **[26-rag-fusion](../26-rag-fusion/)** — fan-out with RRF merge: Send API applied to parallel retrieval queries
- **[6-multi-agent-graph](../6-multi-agent-graph/)** — parallel subgraphs: multiple agents running simultaneously
- **[37-rewoo-agent](../37-rewoo-agent/)** — upfront planning: plan all tool calls before executing any